# Data Mining. Final Project
## 00. Duplicates and train / validation / test split

Dataset: Smoking (body signal of smoking), Kaggle URL: https://www.kaggle.com/datasets/kukuroo3/body-signal-of-smoking

File: `smoking.csv`. Target: `smoking` (1 = smoker, 0 = non-smoker)

Author: Ibragimova Sabina

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

df = pd.read_csv("../data/raw/smoking.csv")
print("Shape:", df.shape)
display(df.head())
print("\nMissing values total:", df.isnull().sum().sum())
print("Duplicated rows (with ID):", df.duplicated().sum())
print("Duplicated rows (without ID):", df.drop(columns=['ID']).duplicated().sum())

Shape: (55692, 27)


,ID,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),...,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,tartar,smoking
0,0,F,40,155,60,81.3,1.2,1.0,1.0,1.0,...,12.9,1.0,0.7,18.0,19.0,27.0,Y,0,Y,0
1,1,F,40,160,60,81.0,0.8,0.6,1.0,1.0,...,12.7,1.0,0.6,22.0,19.0,18.0,Y,0,Y,0
2,2,M,55,170,60,80.0,0.8,0.8,1.0,1.0,...,15.8,1.0,1.0,21.0,16.0,22.0,Y,0,N,1
3,3,M,40,165,70,88.0,1.5,1.5,1.0,1.0,...,14.7,1.0,1.0,19.0,26.0,18.0,Y,0,Y,0
4,4,F,40,155,60,86.0,1.0,1.0,1.0,1.0,...,12.5,1.0,0.6,16.0,14.0,22.0,Y,0,N,0



Missing values total: 0
Duplicated rows (with ID): 0
Duplicated rows (without ID): 11140


The file has 55,692 rows and 27 columns and no missing values. With the `ID` column there are no duplicates, because every row gets its own ID. Without `ID` there are 11,140 exact copies, which is exactly 20% of the data. That looks like the test part of the Kaggle competition version was added to the file a second time. I also checked the `competition_format` folder from the same archive: 11,139 of its 11,140 test rows are copies of train rows. So we don't use that split. If we kept the duplicates, the same person could end up in train and test, and the test score would be too optimistic.

In [3]:
df = df.drop(columns=['ID']).drop_duplicates().reset_index(drop=True)
print("After removing duplicates:", df.shape)

print("\nUnique values in 'oral':", df['oral'].unique().tolist())
df = df.drop(columns=['oral'])

target_column = 'smoking'
print("\nClass distribution:")
print(pd.DataFrame({'count': df[target_column].value_counts(),
                    'share': df[target_column].value_counts(normalize=True).round(3)}))

After removing duplicates: (44552, 26)

Unique values in 'oral': ['Y']

Class distribution:
         count  share
smoking              
0        28200  0.633
1        16352  0.367


After removing duplicates 44,552 rows are left, which is still above the 40,000 rows recommended in the task. The column `oral` has only one value ("Y") in every row, so it can't help any model and I dropped it. 63.3% of people are non-smokers and 36.7% are smokers. The classes are not balanced, but not extremely, so we use a stratified split and check class weights later in modeling.

In [4]:
train, temp = train_test_split(df, test_size=0.30, random_state=RANDOM_STATE, stratify=df[target_column])
val, test = train_test_split(temp, test_size=0.50, random_state=RANDOM_STATE, stratify=temp[target_column])

for name, part in [('train', train), ('val', val), ('test', test)]:
    print(f"{name}: {part.shape} | share of smokers: {part[target_column].mean():.3f}")
    part.to_csv(f"../data/processed/{name}.csv", index=False)

train: (31186, 25) | share of smokers: 0.367


val: (6683, 25) | share of smokers: 0.367
test: (6683, 25) | share of smokers: 0.367


I split the data 70 / 15 / 15: 31,186 rows for training, 6,683 for validation and 6,683 for test. The share of smokers is 0.367 in all three parts, so stratification worked. Everyone in the team uses these files with the same `RANDOM_STATE = 42`. Preprocessing (scaling, encoding) is fitted on train only. Models are compared on validation, and the test set is used once, for the final model only.